# 08 — Model 2: Image Augmentation + Re-unfrozen layer4

`exp07` (fully-frozen ResNet) succeeded: ROC-AUC 0.693, PR-AUC **0.240** -- the first model in the project to beat `text_only_bert`'s PR-AUC (0.212), completing all 10 epochs with no early stopping (stable, no overfitting collapse).

This follow-up (`configs/exp08_model2_augment_unfreeze4.yaml`) keeps everything that worked in exp07 (weight_decay=0.05, eval_every_n_steps=500, early_stopping_patience=16) and adds two changes together:
1. **`unfreeze_image_blocks: 1`** -- re-enable ResNet's `layer4` (was 0, fully frozen in exp07). 22.8M/150.2M trainable (15.2%), same as Model 2 v1.
2. **Image augmentation** on the training split only (random crop, flip, color jitter -- `src/models/hybrid_fallback.py`'s `_TRAIN_TRANSFORM`; val/test still use the deterministic `_EVAL_TRANSFORM`). Only meaningful once part of the vision backbone is actually training, which is why exp07 didn't need it.

**Goal**: recover layer4's extra capacity (which helped v1 reach ROC-AUC 0.660) without v1's fast overfitting, using augmentation as a direct regularizer instead of just freezing capacity away.

Smoke-tested before this run: confirmed 22.8M trainable params (layer4 back), confirmed the train transform is stochastic (varies across calls) and the eval transform is deterministic (identical across calls), and a full `train.py` loop.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt


## 1. Run training

**Do not run this at the same time as any other GPU/MPS process.**

In [ ]:
!cd .. && python3 -u -m src.training.train --config configs/exp08_model2_augment_unfreeze4.yaml


## 2. Load the logged curves and plot

In [ ]:
run_dir = "../experiments/hybrid_model2_augment_unfreeze4"

train_log = pd.read_csv(f"{run_dir}/train_log.csv")
val_log = pd.read_csv(f"{run_dir}/val_log.csv")

print("Training steps logged:", len(train_log))
print("Validation checks logged:", len(val_log))
val_log


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_log["step"], train_log["loss"])
axes[0].set_xlabel("step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Model 2 (augment+unfreeze4) training loss")

axes[1].plot(val_log["step"], val_log["pr_auc"], marker="o", label="PR-AUC")
axes[1].plot(val_log["step"], val_log["roc_auc"], marker="o", label="ROC-AUC")
axes[1].axhline(0.212, color="gray", linestyle="--", label="text_only_bert PR-AUC (0.212)")
axes[1].axhline(0.240, color="green", linestyle=":", label="Model 2 tuned (exp07) PR-AUC (0.240)")
axes[1].set_xlabel("step")
axes[1].set_title("Model 2 (augment+unfreeze4) validation metrics")
axes[1].legend()

plt.tight_layout()
plt.show()


## 3. Compare against every other model tried

In [ ]:
best_v2 = val_log.loc[val_log["pr_auc"].idxmax()]

results = pd.DataFrame([
    {"model": "tfidf_logreg",        "roc_auc": 0.660, "pr_auc": 0.208},
    {"model": "text_only_bert",       "roc_auc": 0.704, "pr_auc": 0.212},
    {"model": "image_only",           "roc_auc": 0.619, "pr_auc": 0.148},
    {"model": "title_image_frozen",   "roc_auc": 0.619, "pr_auc": 0.147},
    {"model": "siglip2_stage_a",      "roc_auc": 0.651, "pr_auc": 0.167},
    {"model": "siglip2_stage_c_lora", "roc_auc": 0.659, "pr_auc": 0.184},
    {"model": "model1_crossattn_full","roc_auc": 0.625, "pr_auc": 0.161},
    {"model": "model1b_crossattn_lora_only", "roc_auc": 0.643, "pr_auc": 0.179},
    {"model": "model2_hybrid_v1",     "roc_auc": 0.660, "pr_auc": 0.212},
    {"model": "model2_hybrid_tuned (exp07)", "roc_auc": 0.693, "pr_auc": 0.240},
    {"model": "model2_augment_unfreeze4 (exp08)", "roc_auc": best_v2["roc_auc"], "pr_auc": best_v2["pr_auc"]},
]).set_index("model")

results.sort_values("pr_auc", ascending=False)
